# 1 · Read a GeoJSON layer like an analyst

**Question:** what do geometry, properties, units, and provenance tell us before a map is drawn?

The five hand-authored features near Washington, DC are **illustrative**. The heights are invented. This is a safe sandbox for checking GIS assumptions before moving to provider data.

![Stylized cartographic splash](https://jltobias.github.io/JupyterLite-RustGIS/assets/rustgis-splash.png)

## Load and inspect

A GeoJSON FeatureCollection is a list of features. Each feature has geometry and properties. Inspect the schema before choosing a map symbol or analysis tool.

In [ ]:
import json
from pathlib import Path
data = json.loads(Path('data/demo_features.geojson').read_text(encoding='utf-8'))
print(data['type'], data['name'], len(data['features']))
print(json.dumps(data['features'][0], indent=2)[:650])

## Summarize geometry types and attributes

In [ ]:
from collections import Counter
features = data['features']
print('Geometry:', dict(Counter(f['geometry']['type'] for f in features)))
print('Kinds:', dict(Counter(f['properties']['kind'] for f in features)))
print('Fields:', sorted({key for f in features for key in f['properties']}))

## Compute the extent

GeoJSON uses WGS84 longitude then latitude. An extent uses **west, south, east, north**. The code handles points and the vertices of the polygon rings.

In [ ]:
def vertices(feature):
    geometry = feature['geometry']
    if geometry['type'] == 'Point':
        return [geometry['coordinates']]
    if geometry['type'] == 'Polygon':
        return [point for ring in geometry['coordinates'] for point in ring]
    raise ValueError(f"Unsupported geometry: {geometry['type']}")
xy = [point for feature in features for point in vertices(feature)]
west, east = min(p[0] for p in xy), max(p[0] for p in xy)
south, north = min(p[1] for p in xy), max(p[1] for p in xy)
print('bbox:', [west, south, east, north])

## Visual check: schematic point view

This SVG is a quick diagnostic, not a projected basemap. Hover over a symbol to read its name.

In [ ]:
from IPython.display import display, HTML
from viz import points, bars
display(HTML(points(features, 'Teaching feature locations')))

## Compare invented heights

Height is only meaningful for polygon footprints here. Do not treat `height_m=0` on an observation point as a measured zero-height building.

In [ ]:
buildings = [f for f in features if f['properties']['kind'] == 'footprint']
display(HTML(bars([f['properties']['name'] for f in buildings], [f['properties']['height_m'] for f in buildings], 'Illustrative heights', ' m')))

## Validate before handoff

In [ ]:
assert all(-180 <= x <= 180 and -90 <= y <= 90 for x, y in xy)
assert all(f['geometry']['coordinates'][0][0] == f['geometry']['coordinates'][0][-1] for f in buildings)
assert all(f['properties']['height_m'] > 0 for f in buildings)
print('Coordinate ranges, closed rings, and illustrative heights checked.')

## Compare with RustGIS

Open the [2D map](https://jltobias.github.io/JupyterLite-RustGIS/maps/2d.html) and the [official RustGIS app](https://web.rustgis.app/?blank&data=https://jltobias.github.io/JupyterLite-RustGIS/data/demo_features.geojson). Can you find the same feature count and extent? Try RustGIS attribute table and layer symbology.

**Challenge:** change one illustrative height in your browser copy, rerun the chart, and describe which map view makes the change easiest to see. Reset the file after the exercise.

**Sources:** [RustGIS](https://github.com/opengeos/RustGIS), [RustGIS web URL parameters](https://github.com/opengeos/RustGIS/blob/main/docs/installation.md#web-app), [your STAC Browser](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/), [STAC specification](https://stacspec.org/). All hosted data here is public teaching material. Check the provider's license before reusing STAC assets.